In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import sys

# ── Dataset ───────────────────────────────────────────────────────────
class ADE20KDataset(Dataset):
    def __init__(self, root, split='training'):
        self.img_dir  = os.path.join(root, 'images', split)
        self.mask_dir = os.path.join(root, 'annotations', split)
        self.images   = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.jpg')])

        self.image_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        self.mask_transform = transforms.Resize(
            (256, 256),
            interpolation=transforms.InterpolationMode.NEAREST
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        image    = Image.open(img_path).convert('RGB')
        image    = self.image_transform(image)

        mask_name = self.images[idx].replace('.jpg', '.png')
        mask_path = os.path.join(self.mask_dir, mask_name)
        mask      = Image.open(mask_path)
        mask      = self.mask_transform(mask)
        mask      = torch.from_numpy(np.array(mask)).long()
        mask      = mask.clamp(0, 149)

        return image, mask

# ── ConvBlock ─────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        return x

# ── U-Net ─────────────────────────────────────────────────────────────
class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=150):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = ConvBlock(512, 1024)
        self.up4  = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(128, 64)
        self.output = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.output(d1)

# ── Setup ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}", flush=True)

ROOT = '/kaggle/input/datasets/awsaf49/ade20k-dataset/ADEChallengeData2016'

train_dataset = ADE20KDataset(root=ROOT, split='training')
val_dataset   = ADE20KDataset(root=ROOT, split='validation')

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Training images:   {len(train_dataset)}", flush=True)
print(f"Validation images: {len(val_dataset)}", flush=True)

sample_img, sample_mask = train_dataset[0]
print(f"Mask min: {sample_mask.min().item()}", flush=True)
print(f"Mask max: {sample_mask.max().item()}", flush=True)

# ── Model ─────────────────────────────────────────────────────────────
model     = UNet(in_channels=3, num_classes=150).to(device)
loss_fn   = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.5
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}", flush=True)

# ── Training loop ─────────────────────────────────────────────────────
for epoch in range(10):
    model.train()
    epoch_loss  = 0
    num_batches = 0

    for i, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks  = masks.to(device)

        predictions = model(images)
        loss        = loss_fn(predictions, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss  += loss.item()
        num_batches += 1

        if i % 100 == 0:
            print(f"Epoch {epoch+1} batch {i}/{len(train_loader)} — loss: {loss.item():.4f}", flush=True)
            sys.stdout.flush()

    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}", flush=True)
    sys.stdout.flush()

    scheduler.step(avg_loss)

    torch.save(model.state_dict(), f"/kaggle/working/unet_ade20k_epoch{epoch+1}.pt")
    print(f"Checkpoint saved: epoch {epoch+1}", flush=True)

print("Training complete!", flush=True)

Training on: cuda
Training images:   20210
Validation images: 2000
Mask min: 0
Mask max: 149
Model parameters: 31,041,430
Epoch 1 batch 0/2527 — loss: 4.9913
Epoch 1 batch 100/2527 — loss: 3.6762
Epoch 1 batch 200/2527 — loss: 3.1663
Epoch 1 batch 300/2527 — loss: 3.0296
Epoch 1 batch 400/2527 — loss: 4.0220
Epoch 1 batch 500/2527 — loss: 3.0538
Epoch 1 batch 600/2527 — loss: 2.9514
Epoch 1 batch 700/2527 — loss: 2.6991
Epoch 1 batch 800/2527 — loss: 2.7173
Epoch 1 batch 900/2527 — loss: 2.5409
Epoch 1 batch 1000/2527 — loss: 3.8175
Epoch 1 batch 1100/2527 — loss: 2.6118
Epoch 1 batch 1200/2527 — loss: 2.9574
Epoch 1 batch 1300/2527 — loss: 3.2632
Epoch 1 batch 1400/2527 — loss: 3.4689
Epoch 1 batch 1500/2527 — loss: 3.3023
Epoch 1 batch 1600/2527 — loss: 2.1135
Epoch 1 batch 1700/2527 — loss: 2.6754
Epoch 1 batch 1800/2527 — loss: 3.1666
Epoch 1 batch 1900/2527 — loss: 3.2280
Epoch 1 batch 2000/2527 — loss: 1.8978
Epoch 1 batch 2100/2527 — loss: 2.6836
Epoch 1 batch 2200/2527 — loss: 